# Residual connections, and the depth they buy

A network that stops learning past a certain depth, and the same network with shortcuts added. The difference is not subtle.

**Runs on:** CPU — about 5 minutes &nbsp;·&nbsp; **Slides:** [Chapter 9 — ConvNet Architecture Patterns](../../../course-web-slides/ch09/index.html) &nbsp;·&nbsp; **Section:** 02 — Residual connections

---

## The problem: depth without shortcuts

In [ ]:
import keras
from keras import layers
from keras.datasets import mnist
import numpy as np

(x, y), (xt, yt) = mnist.load_data()
x = x.reshape(-1, 28, 28, 1).astype("float32") / 255
xt = xt.reshape(-1, 28, 28, 1).astype("float32") / 255

def plain_net(depth):
    keras.utils.set_random_seed(0)
    i = keras.Input(shape=(28, 28, 1))
    z = layers.Conv2D(32, 3, padding="same", activation="relu")(i)
    for _ in range(depth):
        z = layers.Conv2D(32, 3, padding="same", activation="relu")(z)
    z = layers.GlobalAveragePooling2D()(z)
    o = layers.Dense(10, activation="softmax")(z)
    m = keras.Model(i, o)
    m.compile(optimizer="rmsprop", loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
    return m

results = {}
for d in [2, 8, 20]:
    m = plain_net(d)
    h = m.fit(x[:15000], y[:15000], epochs=6, batch_size=128,
              validation_split=.2, verbose=0)
    results[("plain", d)] = max(h.history["val_accuracy"])
    print(f"plain, {d:2d} extra layers -> val acc {results[('plain', d)]:.4f}")

Expected output:

```
plain,  2 extra layers -> val acc 0.98xx
plain,  8 extra layers -> val acc 0.97xx
plain, 20 extra layers -> val acc 0.1x — 0.6x
```

**Deeper is worse.** Not slightly — the twenty-layer version may fail to train at all. That is the vanishing-gradient problem: the signal has to survive twenty successive multiplications on its way back, and it does not.

## The fix, in one line

In [ ]:
def residual_net(depth):
    keras.utils.set_random_seed(0)
    i = keras.Input(shape=(28, 28, 1))
    z = layers.Conv2D(32, 3, padding="same", activation="relu")(i)
    for _ in range(depth):
        residual = z
        z = layers.Conv2D(32, 3, padding="same", activation="relu")(z)
        z = layers.add([z, residual])          # <- the whole idea
    z = layers.GlobalAveragePooling2D()(z)
    o = layers.Dense(10, activation="softmax")(z)
    m = keras.Model(i, o)
    m.compile(optimizer="rmsprop", loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
    return m

for d in [2, 8, 20]:
    m = residual_net(d)
    h = m.fit(x[:15000], y[:15000], epochs=6, batch_size=128,
              validation_split=.2, verbose=0)
    results[("residual", d)] = max(h.history["val_accuracy"])
    print(f"residual, {d:2d} extra layers -> val acc {results[('residual', d)]:.4f}")

In [ ]:
import matplotlib.pyplot as plt

depths = [2, 8, 20]
plt.figure(figsize=(6.5, 4.2))
plt.plot(depths, [results[("plain", d)] for d in depths], "o-", label="plain")
plt.plot(depths, [results[("residual", d)] for d in depths], "s-", label="residual")
plt.xlabel("extra convolution layers"); plt.ylabel("best validation accuracy")
plt.legend(); plt.title("Residual connections keep deep networks trainable")
plt.show()

`x = layers.add([x, residual])` gives the gradient a path back that **skips** the block entirely. It does not have to survive every multiplication — there is always a route home.

## When the shapes do not match

In [ ]:
# Case 1: the number of filters changes -- project with a 1x1 conv.
inputs = keras.Input(shape=(32, 32, 3))
z = layers.Conv2D(32, 3, activation="relu", padding="same")(inputs)
residual = z
z = layers.Conv2D(64, 3, activation="relu", padding="same")(z)
residual = layers.Conv2D(64, 1)(residual)      # no activation
z = layers.add([z, residual])
print("filters changed:", z.shape)

# Case 2: max pooling downsamples -- match it with strides on the shortcut.
inputs = keras.Input(shape=(32, 32, 3))
z = layers.Conv2D(32, 3, activation="relu", padding="same")(inputs)
residual = z
z = layers.Conv2D(64, 3, activation="relu", padding="same")(z)
z = layers.MaxPooling2D(2, padding="same")(z)
residual = layers.Conv2D(64, 1, strides=2)(residual)
z = layers.add([z, residual])
print("downsampled:    ", z.shape)

> **Note** — The 1×1 projection carries **no activation**. Its job is to change shape, not to compute — putting a nonlinearity on the shortcut defeats the point of having a clean path.

## A reusable block

In [ ]:
def residual_block(x, filters, pooling=False):
    residual = x
    x = layers.Conv2D(filters, 3, activation="relu", padding="same")(x)
    x = layers.Conv2D(filters, 3, activation="relu", padding="same")(x)
    if pooling:
        x = layers.MaxPooling2D(2, padding="same")(x)
        residual = layers.Conv2D(filters, 1, strides=2)(residual)
    elif filters != residual.shape[-1]:
        residual = layers.Conv2D(filters, 1)(residual)
    return layers.add([x, residual])

inputs = keras.Input(shape=(32, 32, 3))
z = layers.Rescaling(1./255)(inputs)
z = residual_block(z, filters=32, pooling=True)
z = residual_block(z, filters=64, pooling=True)
z = residual_block(z, filters=128, pooling=False)
z = layers.GlobalAveragePooling2D()(z)
outputs = layers.Dense(1, activation="sigmoid")(z)
model = keras.Model(inputs, outputs)
model.summary()

This is the shape of every modern ConvNet, and — as chapter 15 shows — of every Transformer block too. **Add, then normalize** is not a vision idea; it is a depth idea.

---

## What to take away

- Past a certain depth, plain stacks stop training — the gradient does not survive the trip back.
- `add([x, residual])` gives it a path that skips the block.
- Project the shortcut with a **1×1 convolution and no activation** when shapes differ.
- The same pattern reappears in the Transformer block in chapter 15.